In [1]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Setup and File Loading

import pandas as pd

FILE_PATH = '/content/drive/MyDrive/AI351ProjectTest/04-DataAugmentation/downsampled.csv'

# Read the Excel file into a DataFrame, selecting only the required columns
try:
    df = pd.read_csv(FILE_PATH)

    print(f"DataFrame loaded successfully with {len(df)} rows.")
    print(f"\nFirst 5 rows:")
    display(df.head())

except FileNotFoundError:
    print(f"ERROR: File not found at the specified path: {FILE_PATH}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

DataFrame loaded successfully with 23762 rows.

First 5 rows:


,text,label,label_desc,translation,MetricX24_Raw,MetricX24_Scaled_BATAYAN
0,Normally I hate even thinking about this becau...,0,sadness,Karaniwan akong ayaw mag-isip tungkol dito dah...,0.180561,99.277754
1,"You are not alone, trust me. We all face our h...",0,sadness,"Hindi ka nag-iisa, maniwala ka sa akin. Lahat ...",0.221019,99.115925
2,i felt like talking too but i didn t know what...,0,sadness,Gusto kong magsalita rin pero hindi ko alam ku...,0.923421,96.306317
3,My brother is like my best friend. We are clos...,0,sadness,Ang kapatid ko ay parang pinakamahusay kong ka...,-0.688570,100.000000
4,i feel like nobody really care if i m gone eve...,0,sadness,Nararamdaman kong walang tunay na nag-aalala k...,1.094810,95.620761


In [3]:
class_counts = df["label"].value_counts().sort_index()
print("Class distribution in df:")
display(class_counts)

Class distribution in df:


,count
label,
0,6000
1,5362
2,1304
3,2159
4,2509
5,3841
6,2587


In [4]:
# Define desired sample sizes per class label (0–6)
m = {
    0: 200,   # Sadness
    1: 200,   # Joy
    2: 200,   # Love
    3: 200,   # Anger
    4: 200,   # Fear
    5: 200,   # Anxiety
    6: 200    # Stress
}

print("Sampling sizes per class:")
for k, v in m.items():
    print(f"Class {k}: {v} samples")


Sampling sizes per class:
Class 0: 200 samples
Class 1: 200 samples
Class 2: 200 samples
Class 3: 200 samples
Class 4: 200 samples
Class 5: 200 samples
Class 6: 200 samples


In [5]:
import numpy as np

df_subsamples = {}

for label in range(7):

    df_class = df[df["label"] == label]

    if label not in df["label"].unique():
        print(f"Class {label} not in dataset.")
        continue

    sample_size = m[label]

    if sample_size > len(df_class):
        print(f"WARNING: Class {label}: requested {sample_size}, only {len(df_class)} available.")
        sample_size = len(df_class)

    # Random Seed

    seed = 13

    df_sub = df_class.sample(sample_size, random_state=seed)
    df_subsamples[label] = df_sub
    print(f"Class {label}: Subsampled {sample_size} rows.")


Class 0: Subsampled 200 rows.
Class 1: Subsampled 200 rows.
Class 2: Subsampled 200 rows.
Class 3: Subsampled 200 rows.
Class 4: Subsampled 200 rows.
Class 5: Subsampled 200 rows.
Class 6: Subsampled 200 rows.


In [6]:
!nvidia-smi
!pip install transformers accelerate sentencepiece --quiet

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "aisingapore/Gemma-SEA-LION-v3-9B-IT"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16
)

print("SLM loaded successfully.")


Mon Dec  8 02:56:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   28C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.57G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

SLM loaded successfully.


In [7]:
# Gen Y/Z Slang Injection Prompt Function

def build_prompt(text):
    return (
        # Persona (Do not Remove)
        "You are a synthetic data generator specializing in Gen Y and Gen Z slang adaptation. "

        # Indicate what kind of Data Augmentation they will do
        "Transform the Filipino sentence by injecting modern Filipino social-media slang from Gen Y/Z (e.g., 'lol', 'lmao', 'legit', 'fr', 'amp', 'sanaol', 'char', 'oomf', 'lowkey', 'highkey'). "

        # Additional rules
        "Maintain naturalness: slang must sound like authentic social media usage. Preserve meaning, tone, and intent. Do not overdo it."

        # One-Shot Example
        "Example: 'Ang lungkot naman.' becomes: 'Ang lungkot naman fr' "

        # Prompt to minimize information loss (Do not Remove)
        "Do NOT distort the meaning. Only enhance the style with slang. "

        # Prompt to only output the needed data (Do not Remove)
        "Do NOT explain or give commentary. Output only the revised sentence.\n\n"

        f"Text: {text}\n"
        "Augmented:"
    )


In [8]:
def augment_batch(text_list, batch_size=16):
    augmented = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]

        prompts = [build_prompt(t) for t in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=True,
                temperature=0.7,
                top_p=0.9
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # extract SLM output
        clean = []
        for txt in decoded:
            if "Augmented:" in txt:
                txt = txt.split("Augmented:")[-1].strip()
            clean.append(txt)

        augmented.extend(clean)

    return augmented

In [9]:
all_augmented = []

batch_size = 16

for label, df_sub in df_subsamples.items():
    print(f"\n=== Processing subsample {label} ===")

    tagalog_list = df_sub["translation"].tolist()
    total = len(tagalog_list)

    augmented = []
    next_progress = 5

    for i in range(0, total, batch_size):
        batch = tagalog_list[i:i + batch_size]
        batch_aug = augment_batch(batch, batch_size=batch_size)
        augmented.extend(batch_aug)

        percent = ((i + len(batch)) / total) * 100
        if percent >= next_progress:
            print(f"Subsample {label}: {next_progress}% complete")
            next_progress += 5

    df_aug = df_sub.copy()
    df_aug["translation"] = augmented   # overwrite

    all_augmented.append(df_aug)

print("\nAll subsamples processed successfully.")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



=== Processing subsample 0 ===
Subsample 0: 5% complete
Subsample 0: 10% complete
Subsample 0: 15% complete
Subsample 0: 20% complete
Subsample 0: 25% complete
Subsample 0: 30% complete
Subsample 0: 35% complete
Subsample 0: 40% complete
Subsample 0: 45% complete
Subsample 0: 50% complete
Subsample 0: 55% complete
Subsample 0: 60% complete
Subsample 0: 65% complete

=== Processing subsample 1 ===
Subsample 1: 5% complete
Subsample 1: 10% complete
Subsample 1: 15% complete
Subsample 1: 20% complete
Subsample 1: 25% complete
Subsample 1: 30% complete
Subsample 1: 35% complete
Subsample 1: 40% complete
Subsample 1: 45% complete
Subsample 1: 50% complete
Subsample 1: 55% complete
Subsample 1: 60% complete
Subsample 1: 65% complete

=== Processing subsample 2 ===
Subsample 2: 5% complete
Subsample 2: 10% complete
Subsample 2: 15% complete
Subsample 2: 20% complete
Subsample 2: 25% complete
Subsample 2: 30% complete
Subsample 2: 35% complete
Subsample 2: 40% complete
Subsample 2: 45% comple

In [10]:
df_final = pd.concat(all_augmented, ignore_index=True)

print("Final dataset size:", len(df_final))
df_final.head()

Final dataset size: 1400


,text,label,label_desc,translation,MetricX24_Raw,MetricX24_Scaled_BATAYAN
0,startv kaderiminoyuntv it s also crucial the s...,0,sadness,"Mahalaga din 'tong pag-aaral ng manunulat, san...",1.222364,95.110542
1,"So many events in the past, that should have n...",0,sadness,"Grabe, may mga nangyayari sa past, legit wala ...",0.960396,96.158417
2,i don t know why i feel disheartened about the...,0,sadness,Hindi ko alam kung bakit nababahala ako tungko...,-0.306819,100.000000
3,poison for real i had a c section they gave me...,0,sadness,"Nakakalason talaga, nag-cesarean ako tapos ibu...",0.316834,98.732666
4,i keep these things predominantly for fix func...,0,sadness,"Pinapanatili ko to, mga, pangunahin para sa mg...",0.422864,98.308545


In [11]:
OUTPUT_PATH = "/content/drive/MyDrive/AI351ProjectTest/04-DataAugmentation/Augmentation03.csv"

df_final.to_csv(OUTPUT_PATH, index=False)

print("Saved final augmented dataset to:")
print(OUTPUT_PATH)

Saved final augmented dataset to:
/content/drive/MyDrive/AI351ProjectTest/04-DataAugmentation/Augmentation03.csv


In [12]:
df_aug.head()

,text,label,label_desc,translation,MetricX24_Raw,MetricX24_Scaled_BATAYAN
19270,Only 7 years ago at the age of 9 I think I pea...,6,stress,"Noong pitong taon lang ata, siyam pa ako, akal...",-0.879313,100.000000
19040,I work as a security guard at a busy office bu...,6,stress,Nagtatrabaho ako bilang bantay sa isang abalan...,-0.730642,100.000000
19038,Oh fucking boy! My ex boyfriend made me think ...,6,stress,"Ay, p*tang ina! Lmao, akala ko si Prinseng Cha...",-0.844667,100.000000
21697,Stressed about speaking a second language IÃÂ...,6,stress,Na-stress ako sa pagsasalita ng pangalawang wi...,0.278329,98.886683
22229,Learned helplessness I just looked up why I ge...,6,stress,"Natutunan kong kawalan ng pag-asa, kanina ko l...",0.295907,98.816372
